# L10 · 点积与投影 — a·b = |a||b|cosθ，为什么相似度 = 点积 ÷ 积范数

**目标**：点积 = 逐元素相乘再求和；它能衡量两个向量“有多像”。

**为什么对 Aurora 重要**：`dft()` 里每个频点 `X[k]` 就是信号数组与复指数序列的点积；注意力机制的 QK 打分同理。

## 本课剧情：测试两个方向有多合拍

点积 `a·b = Σaᵢbᵢ` 把两个向量压缩成一个标量，衡量它们方向有多一致。几何上它等于 `|a||b|cosθ`——两向量夹角为 90° 时点积为零，方向相同时最大，反向时最小。本节先算代数公式，再用余弦相似度归一化到 [-1, 1]，最后连到 DFT 的频点计算。

## 1. 点积的定义

点积 `a·b = Σ aᵢbᵢ`：把两个同维向量压缩成一个标量，逐元素相乘后求和。例：`a=[1,2,3]`，`b=[4,5,6]` → 1×4 + 2×5 + 3×6 = **32**。

几何上，`a·b = |a|·|b|·cosθ`，其中 θ 是两向量的夹角。方向完全相同时 cosθ=1，点积取最大正值；垂直时 cosθ=0，点积为零；反向时 cosθ=-1，点积取最大负值。这个公式不是巧合——它把「方向对齐程度」编码成一个数。

Aurora 的 `dft()` 用的正是这一性质：将信号数组与第 k 个复指数序列做点积，得到频点 `X[k]`。频率匹配时两列数据高度对齐、点积最大；频率不匹配时两列数据近似正交、点积趋近于零，实现频率「挑选」。

## 符号入口：先看形状，再看运算

点积的两个输入都必须是同维向量 `(n,)`，输出是标量。每次遇到新运算，先确认输入的形状匹配，再写计算。

In [ ]:
import numpy as np
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print('逐元素相乘:', a * b)        # [4 10 18]
print('点积(求和):', np.dot(a, b)) # 32
print('也可写作 a @ b =', a @ b)

## 动手观察：两个向量，三种角度关系

运行下面的代码，注意 `np.dot(a, b)` 的正负号如何随两向量夹角变化：方向相同时为正，垂直时为零，反向时为负。这就是点积作为“对齐程度”的直观含义。

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
A = np.array([[2.0, 0.0],
              [0.0, 0.5]])

print('v =', v, 'shape =', v.shape)
print('A =')
print(A)
print('A shape =', A.shape)
print('A @ v =', A @ v)
print('向量长度 ||v|| =', np.linalg.norm(v))


## 代码实验：遍历几个向量，观察点积如何响应方向变化

固定向量 `a`，依次与不同方向的 `b` 做点积，观察当 `b` 从同向转到垂直再转到反向时，点积从正数降到零再到负数。

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
vectors = [np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([1.0, 1.0]), np.array([2.0, -1.0])]

print('A =')
print(A)
for v in vectors:
    out = A @ v
    print(f'v={v} -> A@v={out}')


## 2. 余弦相似度：方向有多一致（-1~1，越接近1越像）

`cos = (a·b) / (|a| · |b|)`，其中 `|a|` 是向量长度（下一课细讲）。

## 3. ✏️ 你的任务：实现 `cosine_similarity`

**推理路线**：
1. 公式 `cos = (a·b) / (|a|·|b|)` 把点积除以两个向量的长度之积，消掉大小只留方向——无论向量多长，同向结果始终为 1。
2. 分子是点积 `Σ aᵢbᵢ`，直接用 `np.dot(a, b)` 计算。
3. 分母 `|a|·|b|` 是两个 L2 范数之积（下节推导）；暂时用 `np.linalg.norm()`。
4. 边界验证：同向向量 → 1；垂直向量 → 0；反向向量 → -1。

**参考输入输出**：`a=[1,2,3]`，`b=[4,5,6]` → 点积 = 32，|a|=√14≈3.742，|b|=√77≈8.775，余弦 ≈ **0.974**

<details><summary>点击查看参考实现</summary>

```python
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
```

</details>

### 写 `cosine_similarity` 前明确三件事

- 输入：`a`、`b`，形状均为 `(n,)`，维度必须相同
- 关键步骤：分子 `np.dot(a, b)`，分母 `np.linalg.norm(a) * np.linalg.norm(b)`
- 返回：标量，范围 [-1, 1]；垂直向量应得 0，同向应得 1

In [ ]:
def cosine_similarity(a, b):
    # ✏️ TODO: 返回 a 与 b 的余弦相似度
    return None  # ← 改这里


In [ ]:
song1 = np.array([5.0, 0.0, 3.0, 0.0])  # 假装是歌曲特征向量
song2 = np.array([4.0, 0.0, 2.0, 0.0])  # 和 song1 很像
song3 = np.array([0.0, 5.0, 0.0, 4.0])  # 风格不同
print('sim(1,2) =', round(cosine_similarity(song1, song2), 3), '(应接近 1, 很像)')
print('sim(1,3) =', round(cosine_similarity(song1, song3), 3), '(应接近 0, 不像)')
assert cosine_similarity(song1, song2) > cosine_similarity(song1, song3)
print('\n✅ 通过：你做出了推荐系统的核心度量。')

**🔗 Aurora 连接**：Music Core 的"猜你喜欢"= 找余弦相似度最高的歌。DFT 里 `X[k]` = 信号 · 第 k 个频率的复指数，也是点积。

## 🎨 图示：内积(→标量) 与 外积(→秩1矩阵)

In [ ]:
from laviz import style, vec_times_vec
style()
vec_times_vec([1,2,3],[4,5,6]);

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 参数实验：只改一个旋钮

取 `t = np.linspace(0, 1, 1000)`，令 `a = np.sin(2*np.pi*1*t)`（1 Hz），`b = np.sin(2*np.pi*2*t)`（2 Hz）——不同频率。计算 `np.dot(a, b)`，结果接近 0：不同频率的正弦波互相正交。再令 `b = np.sin(2*np.pi*1*t)`（同频率），点积变为正数；令 `b = 2 * np.sin(2*np.pi*1*t)`，点积翻倍——点积正比于振幅乘积。DFT 的 `X[k]` 正是利用这个性质：与第 k 个频率的复指数做点积，匹配频率的幅度最大，其他频率因正交性消去。

In [ ]:
A = np.array([[2.0, 1.0], [0.0, 1.0]])
probes = np.array([[1,0], [0,1], [1,1], [-1,2]], dtype=float)
print('矩阵 A 会怎样移动这些向量？')
for v in probes:
    out = A @ v
    print(f'{v} -> {out} | 长度 {np.linalg.norm(v):.2f} -> {np.linalg.norm(out):.2f}')


## 本课收束

`dot_product(a, b)` 现在能给出 `Σaᵢbᵢ`，`cosine_similarity(a, b)` 能把结果归一化到 [-1, 1]。Aurora 的 `dft()` 函数里，每个 `X[k]` 的计算就是信号数组与复指数序列的点积，和本节的代数结构完全相同。下一节讲范数，它是余弦相似度分母的来源，也是衡量向量“大小”的工具。

In [ ]:
# 小检查：先看 shape，再做运算
import numpy as np

a = np.array([1.0, 2.0])
b = np.array([3.0, 4.0])
A = np.array([[1.0, 2.0], [0.0, 1.0]])

print('a.shape =', a.shape)
print('b.shape =', b.shape)
print('A.shape =', A.shape)
print('a + b =', a + b)
print('A @ a =', A @ a)
print('a dot b =', float(a @ b))
